In [2]:
import seisbench
import seisbench.models as sbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import sys
import pickle
import bz2
from pathlib import Path
from src  import (
    set_plot_style,
)
colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


In [3]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define all paths from project root with DATA_TYPE subdirectories
METADATA_IMPORT = PROJECT_ROOT / "data" / "raw"

In [5]:
"""
Script per esplorare il file CSV dei metadata di INSTANCE.

Questo script:
1. Apre il file CSV (gestendo compressione bz2 se necessario)
2. Mostra le prime righe
3. Lista tutte le colonne disponibili
4. Cerca eventi vicini alla data del tuo evento (2024-12-09)
5. Identifica le colonne relative ai picks P e S
"""

# ==============================================================================
# CONFIGURAZIONE
# ==============================================================================

# Modifica questo path con il percorso dove hai salvato il file scaricato
FILE_PATH = Path(METADATA_IMPORT / "metadata_Instance_events_v3.csv.bz2")


# ==============================================================================
# FUNZIONE DI APERTURA (gestisce sia .csv che .csv.bz2)
# ==============================================================================

def open_instance_metadata(file_path):
    """
    Apre il file metadata di INSTANCE, gestendo compressione bz2.
    
    Parameters
    ----------
    file_path : Path
        Path al file scaricato
        
    Returns
    -------
    df : pd.DataFrame
        DataFrame con i metadata
    """
    
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    print(f"Opening file: {file_path}")
    print(f"File size: {file_path.stat().st_size / 1e6:.1f} MB")
    
    # Determina se è compresso
    if file_path.suffix == '.bz2':
        print("Detected bz2 compression, decompressing...")
        df = pd.read_csv(file_path, compression='bz2', low_memory=False)
    else:
        df = pd.read_csv(file_path, low_memory=False)
    
    return df


# ==============================================================================
# ESPLORAZIONE
# ==============================================================================

print("=" * 80)
print("INSTANCE METADATA EXPLORATION")
print("=" * 80)

# Se vuoi testare senza scaricare, usa questo comando per vedere se il file esiste
print(f"\nChecking if file exists: {FILE_PATH}")
print(f"Exists: {FILE_PATH.exists()}")

if not FILE_PATH.exists():
    print("\n⚠️  File not found!")
    print("\nPlease:")
    print("1. Download the file from: http://doi.org/10.13127/instance/eventsmetadata.3")
    print("2. Update FILE_PATH in this script with the correct path")
    print("\nExample:")
    print("  FILE_PATH = Path('~/Downloads/eventsmetadata.csv.bz2').expanduser()")
else:
    # Apri il file
    df = open_instance_metadata(FILE_PATH)
    
    print(f"\n{'=' * 80}")
    print("DATASET OVERVIEW")
    print("=" * 80)
    
    print(f"\nShape: {df.shape}")
    print(f"  Rows (records): {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    # Lista TUTTE le colonne
    print(f"\n{'=' * 80}")
    print("ALL COLUMNS")
    print("=" * 80)
    
    for i, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        n_missing = df[col].isna().sum()
        pct_missing = 100 * n_missing / len(df)
        
        print(f"{i:3d}. {col:40s} | {str(dtype):10s} | "
              f"unique: {n_unique:8,} | missing: {pct_missing:5.1f}%")
    
    # Identifica colonne che potrebbero contenere picks
    print(f"\n{'=' * 80}")
    print("COLUMNS RELATED TO PHASE PICKS")
    print("=" * 80)
    
    pick_keywords = ['pick', 'arrival', 'phase', 'p_', 's_', 'time', 'onset']
    pick_columns = [col for col in df.columns 
                    if any(kw in col.lower() for kw in pick_keywords)]
    
    if pick_columns:
        print(f"\nFound {len(pick_columns)} columns related to picks:")
        for col in pick_columns:
            n_valid = df[col].notna().sum()
            print(f"  - {col:40s} | {n_valid:8,} valid values")
    else:
        print("\n⚠️  No obvious pick columns found")
        print("The file might use different naming conventions")
    
    # Mostra le prime righe
    print(f"\n{'=' * 80}")
    print("FIRST 5 ROWS")
    print("=" * 80)
    
    print("\n", df.head())
    
    # Cerca colonne con date/timestamps
    print(f"\n{'=' * 80}")
    print("DATE/TIME COLUMNS")
    print("=" * 80)
    
    time_columns = [col for col in df.columns 
                    if 'time' in col.lower() or 'date' in col.lower()]
    
    if time_columns:
        print(f"\nFound {len(time_columns)} time-related columns:")
        for col in time_columns:
            print(f"\n  {col}:")
            print(f"    Sample values:")
            sample = df[col].dropna().head(3)
            for val in sample:
                print(f"      {val}")
    
    # Cerca eventi vicini alla tua data (2024-12-09)
    print(f"\n{'=' * 80}")
    print("SEARCHING FOR YOUR EVENT (2024-12-09)")
    print("=" * 80)
    
    # Identifica la colonna con la data dell'evento
    event_time_cols = [col for col in df.columns 
                       if 'event' in col.lower() and 'time' in col.lower()]
    
    if not event_time_cols:
        # Prova con 'origin_time' o simili
        event_time_cols = [col for col in df.columns 
                          if 'origin' in col.lower() and 'time' in col.lower()]
    
    if event_time_cols:
        time_col = event_time_cols[0]
        print(f"\nUsing column: {time_col}")
        
        # Converti in datetime
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        
        # Cerca eventi nel dicembre 2024
        mask_dec_2024 = (df[time_col].dt.year == 2024) & (df[time_col].dt.month == 12)
        events_dec = df[mask_dec_2024]
        
        print(f"\nEvents in December 2024: {len(events_dec)}")
        
        if len(events_dec) > 0:
            print("\nEvents found:")
            display_cols = [time_col] + [col for col in df.columns 
                                         if 'station' in col.lower() 
                                         or 'magnitude' in col.lower() 
                                         or 'lat' in col.lower() 
                                         or 'lon' in col.lower()][:5]
            print(events_dec[display_cols].head(20))
            
            # Cerca specificamente il 9 dicembre
            mask_dec_9 = (df[time_col].dt.year == 2024) & \
                        (df[time_col].dt.month == 12) & \
                        (df[time_col].dt.day == 9)
            events_target = df[mask_dec_9]
            
            print(f"\n\nEvents on 2024-12-09: {len(events_target)}")
            
            if len(events_target) > 0:
                print("\n✓ YOUR EVENT MIGHT BE IN THE DATABASE!")
                print("\nSample records:")
                print(events_target[display_cols].head(20))
            else:
                print("\n⚠️  No events found on 2024-12-09")
                print("Your event might be too recent for this version")
        else:
            print("\n⚠️  No events in December 2024")
            print("The database might not include recent events yet")
            
        # Mostra range temporale del database
        print(f"\n{'=' * 80}")
        print("DATABASE TIME RANGE")
        print("=" * 80)
        print(f"\nFirst event: {df[time_col].min()}")
        print(f"Last event: {df[time_col].max()}")
        print(f"Total time span: {(df[time_col].max() - df[time_col].min()).days} days")
        
    else:
        print("\n⚠️  Could not identify event time column")
        print("Please check column names manually")
    
    # Salva un sample per ispezione
    sample_file = FILE_PATH.parent / "instance_metadata_sample.csv"
    df.head(100).to_csv(sample_file, index=False)
    print(f"\n{'=' * 80}")
    print(f"Saved first 100 rows to: {sample_file}")
    print("=" * 80)

print("\n" + "=" * 80)
print("END EXPLORATION")
print("=" * 80)

INSTANCE METADATA EXPLORATION

Checking if file exists: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/raw/metadata_Instance_events_v3.csv.bz2
Exists: True
Opening file: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/raw/metadata_Instance_events_v3.csv.bz2
File size: 286.9 MB
Detected bz2 compression, decompressing...

DATASET OVERVIEW

Shape: (1159249, 115)
  Rows (records): 1,159,249
  Columns: 115

ALL COLUMNS
  1. source_id                                | int64      | unique:   54,008 | missing:   0.0%
  2. station_network_code                     | object     | unique:       18 | missing:   0.0%
  3. station_code                             | object     | unique:      620 | missing:   0.0%
  4. station_location_code                    | float64    | unique:        2 | missing:  99.9%
  5. station_channels                         | object     | unique:        5 | missing:   0.0%
  6. station_latitude_deg                     | float64

In [6]:
# ==============================================================================
# 4. VERIFICA PRESENZA PICKS - TERREMOTO DELL'AQUILA 2009
# ==============================================================================

print("=" * 80)
print("SEARCHING FOR L'AQUILA EARTHQUAKE (2009-04-06)")
print("=" * 80)

# Evento principale L'Aquila
# Data: 6 aprile 2009, ore 01:32 UTC
# Magnitudo: Mw 6.3
# Epicentro: ~42.35°N, 13.35°E
# Profondità: ~8.8 km

# Cerca l'evento principale
mask_aquila_main = (
    (df['source_origin_time'] >= '2009-04-06 01:00:00') & 
    (df['source_origin_time'] <= '2009-04-06 02:00:00') &
    (df['source_latitude_deg'] >= 42.2) & 
    (df['source_latitude_deg'] <= 42.5) &
    (df['source_longitude_deg'] >= 13.2) & 
    (df['source_longitude_deg'] <= 13.5) &
    (df['source_magnitude'] >= 6.0)
)

df_aquila_main = df[mask_aquila_main].copy()

print(f"\nMain shock found: {len(df_aquila_main) > 0}")
print(f"Number of station records: {len(df_aquila_main)}")

if len(df_aquila_main) > 0:
    print("\n" + "-" * 80)
    print("MAIN SHOCK DETAILS")
    print("-" * 80)
    
    # Info evento principale
    event_info = df_aquila_main.iloc[0]
    print(f"\nEvent ID: {event_info['source_id']}")
    print(f"Origin time: {event_info['source_origin_time']}")
    print(f"Magnitude: {event_info['source_magnitude']} {event_info['source_magnitude_type']}")
    print(f"Location: {event_info['source_latitude_deg']:.4f}°N, {event_info['source_longitude_deg']:.4f}°E")
    print(f"Depth: {event_info['source_depth_km']:.2f} km")
    
    # Numero stazioni
    n_stations = df_aquila_main['station_code'].nunique()
    print(f"\nNumber of stations: {n_stations}")
    
    print("\n" + "-" * 80)
    print("PHASE PICKS AVAILABILITY")
    print("-" * 80)
    
    # Verifica presenza picks P e S
    n_p_picks = df_aquila_main['trace_P_arrival_time'].notna().sum()
    n_s_picks = df_aquila_main['trace_S_arrival_time'].notna().sum()
    
    print(f"\nP picks available: {n_p_picks}/{len(df_aquila_main)} ({100*n_p_picks/len(df_aquila_main):.1f}%)")
    print(f"S picks available: {n_s_picks}/{len(df_aquila_main)} ({100*n_s_picks/len(df_aquila_main):.1f}%)")
    
    # Verifica anche i sample indices
    n_p_samples = df_aquila_main['trace_P_arrival_sample'].notna().sum()
    n_s_samples = df_aquila_main['trace_S_arrival_sample'].notna().sum()
    
    print(f"\nP arrival samples available: {n_p_samples}/{len(df_aquila_main)}")
    print(f"S arrival samples available: {n_s_samples}/{len(df_aquila_main)}")
    
    # Mostra alcune stazioni di esempio
    print("\n" + "-" * 80)
    print("SAMPLE STATIONS (first 10)")
    print("-" * 80)
    
    sample_cols = [
        'station_code', 
        'station_network_code',
        'path_hyp_distance_km',
        'trace_P_arrival_time',
        'trace_S_arrival_time',
        'trace_P_arrival_sample',
        'trace_S_arrival_sample'
    ]
    
    print("\n", df_aquila_main[sample_cols].head(10).to_string(index=False))
    
    # Lista tutte le stazioni
    print("\n" + "-" * 80)
    print("ALL STATIONS")
    print("-" * 80)
    
    stations = df_aquila_main.groupby('station_code').agg({
        'station_network_code': 'first',
        'path_hyp_distance_km': 'first',
        'trace_P_arrival_time': lambda x: x.notna().any(),
        'trace_S_arrival_time': lambda x: x.notna().any()
    }).rename(columns={
        'trace_P_arrival_time': 'has_P_pick',
        'trace_S_arrival_time': 'has_S_pick'
    })
    
    print(f"\n{len(stations)} stations with manual picks:")
    print(stations.to_string())
    
    # Verifica qualità picks
    print("\n" + "-" * 80)
    print("PICK QUALITY")
    print("-" * 80)
    
    print(f"\nP pick uncertainty:")
    print(f"  Mean: {df_aquila_main['trace_P_uncertainty_s'].mean():.3f} s")
    print(f"  Median: {df_aquila_main['trace_P_uncertainty_s'].median():.3f} s")
    print(f"  Range: [{df_aquila_main['trace_P_uncertainty_s'].min():.3f}, {df_aquila_main['trace_P_uncertainty_s'].max():.3f}] s")
    
    if n_s_picks > 0:
        print(f"\nS pick uncertainty:")
        print(f"  Mean: {df_aquila_main['trace_S_uncertainty_s'].mean():.3f} s")
        print(f"  Median: {df_aquila_main['trace_S_uncertainty_s'].median():.3f} s")
        print(f"  Range: [{df_aquila_main['trace_S_uncertainty_s'].min():.3f}, {df_aquila_main['trace_S_uncertainty_s'].max():.3f}] s")

else:
    print("\n⚠️  Main shock not found in this exact time window")
    print("\nSearching in broader time window (entire day)...")
    
    # Cerca in finestra più ampia
    mask_aquila_day = (
        (df['source_origin_time'] >= '2009-04-06 00:00:00') & 
        (df['source_origin_time'] <= '2009-04-06 23:59:59') &
        (df['source_latitude_deg'] >= 42.2) & 
        (df['source_latitude_deg'] <= 42.5) &
        (df['source_longitude_deg'] >= 13.2) & 
        (df['source_longitude_deg'] <= 13.5)
    )
    
    df_aquila_day = df[mask_aquila_day].copy()
    
    print(f"\nEvents found on 2009-04-06: {df_aquila_day['source_id'].nunique()}")
    
    if len(df_aquila_day) > 0:
        events = df_aquila_day.groupby('source_id').agg({
            'source_origin_time': 'first',
            'source_magnitude': 'first',
            'station_code': 'count'
        }).rename(columns={'station_code': 'n_records'}).sort_values('source_magnitude', ascending=False)
        
        print("\nTop events by magnitude:")
        print(events.head(10))

# Cerca anche aftershocks significativi
print("\n\n" + "=" * 80)
print("SEARCHING FOR L'AQUILA AFTERSHOCKS (April 2009)")
print("=" * 80)

mask_aquila_sequence = (
    (df['source_origin_time'] >= '2009-04-06') & 
    (df['source_origin_time'] <= '2009-04-30') &
    (df['source_latitude_deg'] >= 42.2) & 
    (df['source_latitude_deg'] <= 42.5) &
    (df['source_longitude_deg'] >= 13.2) & 
    (df['source_longitude_deg'] <= 13.5) &
    (df['source_magnitude'] >= 4.0)
)

df_aquila_sequence = df[mask_aquila_sequence].copy()

print(f"\nEvents M≥4.0 in April 2009: {df_aquila_sequence['source_id'].nunique()}")
print(f"Total station records: {len(df_aquila_sequence)}")

if len(df_aquila_sequence) > 0:
    events_summary = df_aquila_sequence.groupby('source_id').agg({
        'source_origin_time': 'first',
        'source_magnitude': 'first',
        'station_code': 'count'
    }).rename(columns={'station_code': 'n_stations'}).sort_values('source_magnitude', ascending=False)
    
    print("\nAll events M≥4.0:")
    print(events_summary)

print("\n" + "=" * 80)
print("END L'AQUILA SEARCH")
print("=" * 80)

SEARCHING FOR L'AQUILA EARTHQUAKE (2009-04-06)

Main shock found: True
Number of station records: 50

--------------------------------------------------------------------------------
MAIN SHOCK DETAILS
--------------------------------------------------------------------------------

Event ID: 1895389
Origin time: 2009-04-06 01:32:40.400000+00:00
Magnitude: 6.1 Mw
Location: 42.3420°N, 13.3800°E
Depth: 8.30 km

Number of stations: 46

--------------------------------------------------------------------------------
PHASE PICKS AVAILABILITY
--------------------------------------------------------------------------------

P picks available: 50/50 (100.0%)
S picks available: 6/50 (12.0%)

P arrival samples available: 50/50
S arrival samples available: 6/50

--------------------------------------------------------------------------------
SAMPLE STATIONS (first 10)
--------------------------------------------------------------------------------

 station_code station_network_code  path_hyp_dis